# Housing, economy and public services
## Mexicali Urban Liveability Index — `WP08_housing_economy_and_services`

**Lead:** TBC
**Indicators assigned:** 4
**Schema version:** 1.0.0

Housing affordability, land and house prices, jobs, and the quality/adequacy of public services, from census, ENIGH/ENVI, DENUE, cadastral and municipal sources.

Survey-based measures are often only representative at city or AGEB scale. Report at the true scale and flag replication downward rather than implying fine-grained variation.

> New to this project? Work through
> [`00_overview_and_schema.ipynb`](00_overview_and_schema.ipynb)
> first — it carries one indicator end to end. Then read
> [`docs/analyst_guide.md`](../docs/analyst_guide.md).

## How to work through this notebook

For each indicator assigned to you, in this order:

1. **Read the brief.** It reproduces everything the team already
   recorded in the workbook — the draft rationale, the article the
   indicator was adapted from, candidate data sources, and the open
   questions colleagues raised. Do not retype any of it; it is
   already in your metadata stub.
2. **Write the causal pathway sentence** (guide §2.1) and find
   **independent health evidence** for it (§2.2). Do this *before*
   looking for data. Fill in `meta['rationale']`.
3. **Find and document the data** (§3): citation, URL, date
   retrieved, licence, and whether it reaches Condesa.
4. **Compute** at the finest scale your data genuinely support.
   Produce a `DataFrame` with `geo_id` and `value`.
5. **Harmonise** with `uli.harmonise(...)`, label with
   `uli.label(...)`, and **deliver** with
   `uli.write_indicator(...)`.
6. **Look at the map.** Most errors are obvious in ten seconds and
   invisible in a table.

`uli.write_indicator` validates first and refuses to publish a
failing deliverable. While you are still iterating, pass
`allow_failure=True` to write a draft anyway.

Full guidance: [`docs/analyst_guide.md`](../docs/analyst_guide.md).
Schema: [`schema/ULI_output_schema.md`](../schema/ULI_output_schema.md).

## Framing the indicator against health evidence

Every indicator must be justified by evidence of a **meaningful
health or wellbeing benefit**, independent of the liveability
article it was adapted from. Those articles establish that an
indicator is used; they rarely establish that it matters.

Complete this sentence before you compute anything:

> *[what I measure]* changes *[a mechanism]*, which changes *[a
> behaviour or exposure]*, which affects *[a health outcome]*.

For most indicators in this project the behaviour is **walking for
transport**, **walking or recreation in public space**, or
**social contact** — and the exposure is **heat**, **air
pollution** or **injury risk**. Say which, using the vocabulary in
`uli.vocab.HEALTH_PATHWAYS`.

Prefer meta-analyses and systematic reviews, then reputable
guidance (WHO, UN-Habitat, PAHO, Secretaría de Salud), then cohort
studies and natural experiments. Record the **effect size with its
uncertainty**.

**If the evidence supports a different threshold from the one the
workbook proposes, use the evidence-based threshold** and say so in
`threshold_justification`. That is explicitly what the project
wants.

**Mexicali is arid and extremely hot.** Most of this literature
comes from temperate cities. Where the transfer is doubtful — for
example, distance-based walkability thresholds in a city where
summer maxima exceed 45 °C and shade rather than distance is the
binding constraint — record it in `rationale.arid_context`. That is
a contribution, not a caveat.

## When several workbook rows are really one indicator

The workbook harvested indicators article by article, so a single
construct sometimes appears as several rows seen through different
lenses or over different time periods. Air quality is the clearest
case:

| Row | What it is | Lens | Time basis |
|---|---|---|---|
| #292 Air quality | the index value itself | `quality` | `annual_mean` |
| #8 Good air quality | that value against a standard | `quality` | `threshold_share` |
| #293 Days with good air quality | how often the standard is met | `quantity` | `threshold_compliance_days` |
| #173 Days PM2.5 over WHO | the same, for one pollutant | `quantity` | `threshold_exceedance_days` |

These are not four indicators — they are one construct measured
four ways, and computing them separately would mean four
inconsistent methods and four sets of data documentation.

Deliver them as a **measure family**: give every measure the same
`measure_family` slug, and distinguish them with `temporal_basis`
(see `uli.vocab.TEMPORAL_BASES`) and `threshold`. They can still
live under separate workbook ids — the family slug is what tells
the index step, and Reimagina Urbana, that they belong together.

```python
for meta in (meta_292, meta_8, meta_293, meta_173):
    for measure in meta['measures']:
        measure['measure_family'] = 'air_quality'
    meta['data_sources'] = SHARED_SOURCES   # one method, one source
```

The same pattern applies to mean summer temperature versus days
above a comfort threshold (WP02), and to flood extent versus annual
average days of flooding (WP05).

## Condesa coverage is a requirement, not a nicety

The Condesa new development in south-east Mexicali is a project
focus area, and it defeats the usual assumptions:

- about **20%** of it falls outside the previously configured
  study region boundary;
- only **44%** of its area is covered by census manzana polygons,
  so a **manzana-native calculation reaches 33 of the 40
  fraccionamientos, while a `grid_100m`-native one reaches all
  40**;
- it is platted and roaded (43 km of street network in OpenStreetMap
  across 27 of the 40 fraccionamientos) but essentially unbuilt —
  **zero destinations**, and satellite-derived population products
  see almost nobody there.

**One thing is your decision: the native scale.** If your data
allow it, compute on the 100 m grid. That is the difference between
reaching all of Condesa and quietly missing a fifth of it.

Everything else is handled downstream. Population denominators,
the 2030 occupancy scenario and population-weighted exposure
statistics are a reporting-step concern (`uli.exposure`), decided
once for the whole project rather than by each analyst. Urban
fabric and exposure measures — land cover, air quality, heat,
hazards, street infrastructure — are properties of *place*, and
should be computed as such; who lives there is applied later.

Two things to record, though:

- `data_sources[].condesa_coverage` — whether your **source**
  reaches Condesa. Satellite imagery and OSM generally do; a 2020
  census variable or a household survey generally does not.
- `method.condesa_treatment` — what you did about it. Where a
  source does not reach Condesa, mark those rows `no_data` rather
  than omitting them.

The validator treats poor Condesa coverage as an **error**.

---
## Setup

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import uli

# Identify yourself once; it is copied into every deliverable.
ANALYST = {
    'name': 'TODO: your name',
    'email': None,
    'institution': None,
}

print(f'ULI schema version {uli.SCHEMA_VERSION}')
print(f'Reference geographies available: {uli.geography.available()}')

        WORK_PACKAGE = 'WP08_housing_economy_and_services'
        NOTEBOOK = 'notebooks/08_housing_economy_and_services.ipynb'

---
## Your indicators (4)

Each has a brief reproducing what the workbook records,
then three working cells: documentation, calculation,
delivery.

### 289 — Housing affordability

`housing_affordability` · *Housing · Affordable Housing · Housing affordability*

- **Lenses to deliver:** accessibility, quantity
- **Draft rationale (rewrite this):** Affordable housing is essential for liveability and health as high housing costs limit resources for other social determinants of health like food and healthcare.
- **Adapted from:** #22: Alderton (2019), ‘What is the meaning of urban liveability for a city in a low-to-middle-income country? Contextualising liveability for Bangkok, Thailand’; #27: Alzaim (2024), ‘Integrated Framework for Enhancing Liveability and Ecological Sustainability in UAE Communities’
- **Candidate data sources:** "con la ENIGH se puede calcular gasto de alquiler entre ingresos" https://www.inegi.org.mx/programas/enigh/nc/2024/ "Encuesta Nacional de Vivienda (ENVI) 2020" https://www.inegi.org.mx/programas/envi/2020/ " Tipología de vivienda (2024)" from the Actualizaciones del territorio maps https://www.mexicali.gob.mx/sitioimip/geovisor
- **Team notes:** ER: Data will only be available at the city-level from the "Encuesta Nacional de Vivienda (ENVI) 2020" https://www.inegi.org.mx/programas/envi/2020/ We could also use the categories from the " Tipología de vivienda (2024)" from the Actualizaciones del territorio maps https://www.mexicali.gob.mx/sitioimip/geovisor

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_289) at any time to list what is
# still outstanding.
meta_289 = uli.metadata_stub(289, analyst=ANALYST)

# meta_289['rationale']['statement'] = """..."""
# meta_289['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_289['rationale']['arid_context'] = '...'
# meta_289['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_289['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_289)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_289 = 'grid_100m'
METHOD_289 = 'population_weighted_mean'

native_289 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_289 = uli.harmonise(
    native_289,
    native_scale=NATIVE_SCALE_289,
    method=METHOD_289,
)
results_289 = uli.label(
    harmonised_289,
    meta_289,
    measure_id='housing_affordability__accessibility',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_289, meta_289))
# uli.write_indicator(results_289, meta_289)

### 290 — Land and house price

`land_and_house_price` · *Housing · Affordable Housing · Land and house price*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** High housing and land prices are linked to reduced affordability and financial stress, which can negatively impact residents' overall quality of life and residential stability.
- **Adapted from:** #16: Kashi (2025), ‘Spatial analysis and ranking of urban districts based on a comprehensive livability approach: the case of Tehran’
- **Candidate data sources:** "con la ENIGH se puede calcular gasto de alquiler entre ingresos" https://www.inegi.org.mx/programas/enigh/nc/2024/ "Encuesta Nacional de Vivienda (ENVI) 2020" https://www.inegi.org.mx/programas/envi/2020/ " Tipología de vivienda (2024)" from the Actualizaciones del territorio maps https://www.mexicali.gob.mx/sitioimip/geovisor
- **Team notes:** ER: Juan, el catastro tiene costo del lote y vivienda? https://tecmx.sharepoint.com/:f:/s/Mexicali-SIUM/IgCo34MzoO2pQ6n_8_fN7n7yAXbMN8SNG9AALvIBr0s7taI?e=R4YpGb https://tecmx.sharepoint.com/:f:/s/Mexicali-SIUM/IgCIU9bPtEF1RI3-GeohnPsyAV8qQj9UYPAZZ3rpy6S1RZE?e=mcIQXk
- **Open questions raised:** CH: To judge what is "high" and unaffordable may require some interpretation / guidance / use of a threshold (in Australia we us a 30:40 measure of housing affordability stress https://www.ahuri.edu.au/analysis/brief/understanding-3040-indicator-housing-affordability-stress

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_290) at any time to list what is
# still outstanding.
meta_290 = uli.metadata_stub(290, analyst=ANALYST)

# meta_290['rationale']['statement'] = """..."""
# meta_290['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_290['rationale']['arid_context'] = '...'
# meta_290['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_290['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_290)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_290 = 'grid_100m'
METHOD_290 = 'population_weighted_mean'

native_290 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_290 = uli.harmonise(
    native_290,
    native_scale=NATIVE_SCALE_290,
    method=METHOD_290,
)
results_290 = uli.label(
    harmonised_290,
    meta_290,
    measure_id='land_and_house_price__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_290, meta_290))
# uli.write_indicator(results_290, meta_290)

### 1 — Public services

`public_services` · *Social Infrastructure · Public Services · Community Facilities · Public services*

- **Lenses to deliver:** accessibility, quality
- **Draft rationale (rewrite this):** Access to telecommunication infrastructure and mobile network coverage is identified as a vital public service and a major interest for resident liveability.
- **Adapted from:** #8: Onnom (2018), ‘Development of a Liveable City Index (LCI) Using Multi Criteria Geospatial Modelling for Medium Class Cities in Developing Countries’
- **Open questions raised:** CH: is there much variation in access to telecommunications infrastructure in mexicali?

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_1) at any time to list what is
# still outstanding.
meta_1 = uli.metadata_stub(1, analyst=ANALYST)

# meta_1['rationale']['statement'] = """..."""
# meta_1['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_1['rationale']['arid_context'] = '...'
# meta_1['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_1['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_1)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_1 = 'grid_100m'
METHOD_1 = 'population_weighted_mean'

native_1 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_1 = uli.harmonise(
    native_1,
    native_scale=NATIVE_SCALE_1,
    method=METHOD_1,
)
results_1 = uli.label(
    harmonised_1,
    meta_1,
    measure_id='public_services__accessibility',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_1, meta_1))
# uli.write_indicator(results_1, meta_1)

### 112 — Jobs

`jobs` · *Economic Development · Employment · Jobs*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Local employment opportunities allow residents to live and work within the same district, improving work-life balance and reducing the health burdens of long commutes.
- **Adapted from:** #22: Alderton (2019), ‘What is the meaning of urban liveability for a city in a low-to-middle-income country? Contextualising liveability for Bangkok, Thailand’
- **Team notes:** Pues sí uno tiene El valor de fuerza laboral sería mejor con ese y El cociente sobre los que pueden trabajar

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_112) at any time to list what is
# still outstanding.
meta_112 = uli.metadata_stub(112, analyst=ANALYST)

# meta_112['rationale']['statement'] = """..."""
# meta_112['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_112['rationale']['arid_context'] = '...'
# meta_112['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_112['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_112)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_112 = 'grid_100m'
METHOD_112 = 'population_weighted_mean'

native_112 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_112 = uli.harmonise(
    native_112,
    native_scale=NATIVE_SCALE_112,
    method=METHOD_112,
)
results_112 = uli.label(
    harmonised_112,
    meta_112,
    measure_id='jobs__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_112, meta_112))
# uli.write_indicator(results_112, meta_112)

---
## Check what this work package has delivered

In [ ]:
delivered, catalogue = uli.collect()
if len(catalogue):
    display(catalogue)
    print(delivered.groupby(['indicator_code', 'geo_level']).size())
else:
    print('Nothing delivered yet.')

In [ ]:
# Sanity-check a delivered measure on a map before you call it done.
# MEASURE = 'your_indicator_code__quantity'
# LEVEL = 'manzana'
# units = uli.geography.load(LEVEL).merge(
#     delivered.query('measure_id == @MEASURE and geo_level == @LEVEL'),
#     on='geo_id', how='left')
# ax = units.plot(column='value', legend=True, figsize=(11, 8),
#                 missing_kwds={'color': 'lightgrey'})
# condesa = uli.geography.load('condesa_fraccionamiento')
# condesa.boundary.plot(ax=ax, color='red', linewidth=1)
# ax.set_title(MEASURE)
# ax.set_axis_off()